# ✂️ Auto Clipper - Google Colab GPU Backend Server

Notebook ini menjalankan backend **Auto Clipper** di Google Colab. Dengan runtime **T4 GPU**, pembuatan subtitle (transkripsi Whisper) memakai GPU.

Notebook memasang FFmpeg standar dari Ubuntu. Render klip memakai GPU hanya bila runtime Colab mengizinkannya. Bila tidak, render berjalan di prosesor: lebih lambat, tetapi klip tetap jadi. Mana yang Anda dapat tertulis di **Pemeriksaan awal** yang muncul saat server dijalankan di Langkah 4.

Selama proses, file dikerjakan di penyimpanan sementara Colab (`/content/work`). Klip yang sudah jadi dan riwayat proyek disalin ke Google Drive.

---

### 1. Hubungkan Google Drive (Mount Drive)
Hubungkan Google Drive Anda. Klip yang sudah jadi, file subtitle, dan riwayat proyek (`history.db`) disimpan di folder `AutoClipperData` di Drive, jadi tidak hilang saat sesi Colab berakhir. Video asli tidak disimpan di Drive: bila proyek dilanjutkan di sesi baru, video diunduh ulang.

In [ ]:
import os
try:
  import torch
  if not torch.cuda.is_available():
    print("\n=======================================================")
    print("❌ PERINGATAN KRITIS: GPU TIDAK AKTIF / ANDA MEMAKAI CPU!")
    print("\n👉 Pantesan prosesnya lemot banget! Google Colab Anda kehabisan kuota GPU atau salah pengaturan.")
    print("👉 CARA FIX: Klik menu atas \"Runtime\" -> \"Change runtime type\" -> Pilih Hardware accelerator: \"T4 GPU\" lalu Save.")
    print("=======================================================\n")
  else:
    print("✅ GPU T4 Aktif dan siap digunakan!\n")
except Exception:
  pass

from google.colab import drive
drive.mount('/content/drive')

### 2. Pasang Sistem & Dependensi (FFmpeg, Cloudflared & Deno)
Memasang `ffmpeg` untuk pengolah video, `cloudflared` untuk tunnel jaringan aman, dan `deno`, program kecil yang dipakai pengunduh video untuk menjawab pemeriksaan dari YouTube.

Jika Deno gagal dipasang, kotak ini memberi tahu dan langkah berikutnya tetap bisa dijalankan. Akibatnya hanya satu: sebagian video YouTube lebih sering gagal diunduh.

In [ ]:
!apt-get update -qq
!apt-get install -y -qq ffmpeg unzip
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

# Deno: program kecil yang dipakai pengunduh video (yt-dlp) untuk menjawab pemeriksaan dari YouTube.
# Perintah pemasangan resmi: https://docs.deno.com/runtime/getting_started/installation/
# DENO_INSTALL=/usr/local menaruh programnya di /usr/local/bin, yang sudah termasuk PATH,
# jadi server di Langkah 4 langsung menemukannya.
import os, shutil, subprocess

DENO_INSTALL_COMMAND = "curl -fsSL https://deno.land/install.sh | DENO_INSTALL=/usr/local sh"

def versi_deno():
    lokasi = shutil.which("deno")
    if not lokasi:
        return None
    try:
        keluaran = subprocess.run([lokasi, "--version"], capture_output=True, text=True, timeout=30).stdout
        return keluaran.splitlines()[0].strip() if keluaran.strip() else None
    except Exception:
        return None

def pasang_deno():
    versi = versi_deno()
    if versi:
        print(f"Deno sudah terpasang ({versi}).")
        return
    print("Memasang Deno...")
    try:
        hasil = subprocess.run(DENO_INSTALL_COMMAND, shell=True, capture_output=True, text=True, timeout=300)
        detail = ((hasil.stdout or "") + (hasil.stderr or "")).strip()
    except Exception as e:
        detail = f"{type(e).__name__}: {e}"
    if "/usr/local/bin" not in os.environ.get("PATH", "").split(os.pathsep):
        os.environ["PATH"] = "/usr/local/bin" + os.pathsep + os.environ.get("PATH", "")
    versi = versi_deno()
    if versi:
        print(f"Deno terpasang ({versi}).")
        return
    print("PERHATIAN: Deno tidak berhasil dipasang.")
    print("Aplikasi tetap bisa dipakai, tetapi sebagian video YouTube lebih sering gagal diunduh.")
    print("Jalankan ulang kotak ini untuk mencoba lagi. Jika tetap gagal, lanjutkan ke Langkah 3.")
    print("Bila nanti unduhan YouTube gagal, unggah file videonya langsung di aplikasi.")
    print("Detail teknis:", detail[-500:] or "tidak ada keterangan")

pasang_deno()

### 3. Unduh Program Auto Clipper & Dependensi Python
Mengunduh repositori dan memasang seluruh library Python yang dibutuhkan.

Biarkan `BRANCH` berisi `main`. Ganti hanya bila Anda sengaja mencoba versi yang belum dirilis, misalnya `feat/sp0-pipeline-stability`.

In [ ]:
BRANCH = "main" #@param {type:"string"}
BRANCH = BRANCH.strip() or "main"
!git clone -b "$BRANCH" https://github.com/DhimasPH/auto-clipper.git /content/auto-clipper || (cd /content/auto-clipper && git fetch origin "$BRANCH" && git checkout "$BRANCH" && git pull origin "$BRANCH")
%cd /content/auto-clipper
!echo "Versi yang dipakai: $(git rev-parse --abbrev-ref HEAD) @ $(git rev-parse --short HEAD)"
!pip install -r backend/requirements.txt uvicorn pyngrok
!pip install -U yt-dlp
# Pustaka GPU untuk pembuatan subtitle (Whisper). Tanpa ini subtitle dibuat dengan prosesor dan bisa makan lebih dari satu jam.
!pip install nvidia-cublas-cu12 "nvidia-cudnn-cu12==9.*"
# Pelacak wajah Teliti (MediaPipe 1.2.3 + face_landmarker.task versi dan SHA-256 terkunci).
!python -c 'from backend.addons import install_colab_mediapipe; print("Pelacak teliti siap:", install_colab_mediapipe())'

### 4A. Jalankan Backend (Pilihan A: Cloudflare Tunnel - Punya Domain Sendiri)
Masukkan **Cloudflare Tunnel Token** dan **API Secret Token (Password Rahasia)** pada formulir di bawah ini, lalu klik tombol **Play (▶️)**.

Sebelum server menyala, kotak ini menampilkan **Pemeriksaan awal**: keadaan GPU, Google Drive, ruang penyimpanan, memori, dan pengunduh video. Baris bertanda `PERHATIAN` tidak menghentikan server. Baris bertanda `MASALAH` harus diperbaiki dulu, karena proses video tidak akan dimulai selama masalah itu ada.

In [ ]:
%cd /content/auto-clipper
#@title Jalankan Server via Cloudflare Tunnel
CLOUDFLARE_TUNNEL_TOKEN = "" #@param {type:"string"}
API_SECRET_TOKEN = "" #@param {type:"string"}

import os
# Klip jadi dan riwayat disimpan di Google Drive. Selama proses, file dikerjakan di disk lokal Colab.
os.environ["AUTO_CLIPPER_CLOUD_MODE"] = "1"
os.environ["AUTO_CLIPPER_WORKSPACE"] = "/content/drive/MyDrive/AutoClipperData"
os.environ["AUTO_CLIPPER_WORK_DIR"] = "/content/work"

!python scripts/colab_preflight.py
!python backend/colab_api.py --cloudflare-token "$CLOUDFLARE_TUNNEL_TOKEN" --api-token "$API_SECRET_TOKEN"

### 4B. Jalankan Backend (Pilihan B: Ngrok Tunnel - 100% Gratis Tanpa Domain)
Jika Anda tidak memiliki domain pribadi di Cloudflare, gunakan formulir di bawah ini dengan **Ngrok Authtoken** (Daftar gratis di ngrok.com).

Sebelum server menyala, kotak ini menampilkan **Pemeriksaan awal**: keadaan GPU, Google Drive, ruang penyimpanan, memori, dan pengunduh video. Baris bertanda `PERHATIAN` tidak menghentikan server. Baris bertanda `MASALAH` harus diperbaiki dulu, karena proses video tidak akan dimulai selama masalah itu ada.

In [ ]:
%cd /content/auto-clipper
#@title Jalankan Server via Ngrok Tunnel
NGROK_AUTHTOKEN = "" #@param {type:"string"}
API_SECRET_TOKEN = "" #@param {type:"string"}

from pyngrok import ngrok
import os

if NGROK_AUTHTOKEN.strip():
    ngrok.set_auth_token(NGROK_AUTHTOKEN.strip())

public_url = ngrok.connect(8000).public_url
print(f"\n=========================================")
print(f"🚀 PUBLIC BACKEND URL ANDA: {public_url}")
print(f"=========================================\n")

os.environ["API_SECRET_TOKEN"] = API_SECRET_TOKEN.strip()
os.environ["AUTO_CLIPPER_DEV_TOKEN"] = API_SECRET_TOKEN.strip()
os.environ["AUTO_CLIPPER_WEB_TOKEN"] = API_SECRET_TOKEN.strip()
os.environ["AUTO_CLIPPER_CLOUD_MODE"] = "1"
# Klip jadi dan riwayat disimpan di Google Drive. Selama proses, file dikerjakan di disk lokal Colab.
os.environ["AUTO_CLIPPER_WORKSPACE"] = "/content/drive/MyDrive/AutoClipperData"
os.environ["AUTO_CLIPPER_WORK_DIR"] = "/content/work"

try:
    import nvidia.cublas.lib
    import nvidia.cudnn.lib
    os.environ["LD_LIBRARY_PATH"] = os.path.dirname(nvidia.cublas.lib.__file__) + ":" + os.path.dirname(nvidia.cudnn.lib.__file__) + ":" + os.environ.get("LD_LIBRARY_PATH", "")
except Exception:
    os.environ["LD_LIBRARY_PATH"] = "/usr/local/cuda/lib64:" + os.environ.get("LD_LIBRARY_PATH", "")

!python scripts/colab_preflight.py
!python -m uvicorn backend.main:app --host 0.0.0.0 --port 8000